# Objektspårning och klassificering med AI 🎯

Du har tränat CNN:er och kört en live-kamera. Nu lägger vi till ett nytt lager: **objektspårning**.

## Tre lager i pipelinen

| Steg | Vad det gör | Vem gör det |
|------|-------------|-------------|
| **Detektion** | Hittar objekt i *ett frame* — ger bounding boxes | YOLOv8 |
| **Tracking** | Kopplar ihop boxes *över tid* — samma objekt, samma ID | ByteTrack |
| **Klassificering** | Vad exakt *är* det? — dina egna kategorier | MobileNetV2 / CLIP |

## Vad du behöver
- Ett Google-konto med Drive
- En kamera **eller** en videofil att ladda upp
- Kör alla celler uppifrån och ned

## Vad du väljer
- **Grunduppgift:** Träna en egen klassificerare (MobileNetV2) och plugga in den
- **Utmaningsuppgift:** Testa zero-shot klassificering med CLIP — inga träningsbilder krävs

---
## Del 1: Installation och laddning

Kör cellen nedan för att installera nödvändiga bibliotek. Det tar ~1 minut.

In [ ]:
!pip install ultralytics trackers supervision --quiet

In [ ]:
import os
import base64
import json
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

import supervision as sv
from ultralytics import YOLO
from trackers import ByteTrackTracker

from IPython.display import display, HTML
from google.colab import files, output as colab_output

print("Alla bibliotek importerade! ✓")

### Ladda detektor och tracker

YOLOv8n är den minsta och snabbaste varianten — bra för Colab.
Den är förtränad på 80 vanliga objekt (COCO-dataset): person, flaska, stol, laptop, bil...

In [ ]:
# Ladda YOLOv8n — laddas ner automatiskt första gången (~6 MB)
yolo = YOLO('yolov8n.pt')

# Skapa tracker — håller reda på vilka objekt som är vilka över tid
tracker = ByteTrackTracker()

# Annotators från supervision
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=40)

# Klassnamn som YOLO känner till
YOLO_KLASSER = yolo.model.names  # dict: {0: 'person', 1: 'bicycle', ...}

print(f"YOLO laddad ✓  ({len(YOLO_KLASSER)} klasser)")
print(f"Exempel på klasser YOLO känner igen:")
print(", ".join(list(YOLO_KLASSER.values())[:20]))

---
## Del 2: Välj input

Ändra `INPUT` nedan och kör cellen.

| Värde | Vad som händer |
|-------|----------------|
| `'video'` | Du får ladda upp en videofil (.mp4, .avi, .mov) |
| `'kamera'` | Din webcam startar direkt i nästa del |

In [ ]:
INPUT = 'video'  # Ändra till 'kamera' för live-kamera

VIDEO_PATH  = None   # fylls i automatiskt vid upload
OUTPUT_PATH = '/content/annotated_output.mp4'

if INPUT == 'video':
    print("Ladda upp en videofil (mp4, avi, mov):")
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = list(uploaded.keys())[0]
        print(f"Video uppladdad: {VIDEO_PATH} ✓")
    else:
        print("Ingen fil uppladdad.")
elif INPUT == 'kamera':
    print("Kamera-läge valt. Kör nästa del för att starta kameran. ✓")
else:
    raise ValueError(f"INPUT måste vara 'video' eller 'kamera', fick: '{INPUT}'")

---
## Del 3: Kör bas-pipeline

Kör cellen nedan. YOLO detekterar objekt, ByteTrack håller koll på vilka de är över tid,
och supervision ritar ut bounding boxes med ID:n och rörelsebanor (trajektorier).

Om du kör video sparas resultatet som ny videofil. Om du kör kamera visas ett live-flöde.

In [ ]:
def bygg_etikett(tracker_id, class_id, extra_label=None):
    """Bygger textetiketten som visas ovanför varje bounding box."""
    yolo_klass = YOLO_KLASSER.get(int(class_id), '?')
    etikett = f"#{tracker_id} {yolo_klass}"
    if extra_label:
        etikett += f" | {extra_label}"
    return etikett

In [ ]:
def kör_video_pipeline(video_path, output_path, extra_classifier=None):
    """
    Kör YOLO + ByteTrack på en videofil och sparar annoterat resultat.

    Args:
        video_path:        sökväg till input-video
        output_path:       sökväg till output-video
        extra_classifier:  valfri funktion(crop_bgr) -> str, t.ex. MobileNetV2 eller CLIP
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Kunde inte öppna video: {video_path}")

    fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tot  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

    # Återställ trackern för varje ny video
    global tracker
    tracker = ByteTrackTracker()

    frame_nr = 0
    print(f"Bearbetar {tot} frames ({w}×{h} @ {fps:.1f} fps)...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detektion
        results    = yolo(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)

        # Tracking
        tracked = tracker.update(detections)

        # Bygg etiketter
        etiketter = []
        for tid, cid, box in zip(tracked.tracker_id, tracked.class_id, tracked.xyxy):
            extra = None
            if extra_classifier is not None:
                x1, y1, x2, y2 = map(int, box)
                crop = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    extra = extra_classifier(crop)
            etiketter.append(bygg_etikett(tid, cid, extra))

        # Rita ut
        frame = trace_annotator.annotate(frame, tracked)
        frame = box_annotator.annotate(frame, tracked)
        frame = label_annotator.annotate(frame, tracked, labels=etiketter)

        out.write(frame)
        frame_nr += 1
        if frame_nr % 30 == 0:
            print(f"  Frame {frame_nr}/{tot}...")

    cap.release()
    out.release()
    print(f"Klar! Sparad till: {output_path}")

In [ ]:
if INPUT == 'video' and VIDEO_PATH:
    kör_video_pipeline(VIDEO_PATH, OUTPUT_PATH)

    # Visa annoterad video direkt i notebook
    with open(OUTPUT_PATH, 'rb') as f:
        video_b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'''
        <video controls width="640">
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
    '''))
elif INPUT == 'kamera':
    print("Du kör i kamera-läge — hoppa till kamera-cellen nedan.")